# Lesson 10a: The Transformer — Theory

9a derived scaled dot-product attention as a differentiable soft lookup
and verified it against PyTorch; 9b showed one attention layer
substantially outperforming a fixed-vector bottleneck on a real task.
The Transformer (Vaswani et al., 2017) builds an entire architecture
around that one operator — **no recurrence at all** — by running several
attention computations in parallel (**multi-head attention**), injecting
the position information attention itself does not have (**positional
encodings**), stacking the result into residual, normalised blocks (**the
Transformer block**), and, for autoregressive generation, hiding the
future from every position during training (**causal masking**). This
lesson derives and implements every one of those four pieces from
scratch, and closes by counting a block's parameters by hand and
checking the count against a real implementation.

By the end of this notebook you will have:
- derived and implemented **multi-head self-attention** from scratch,
  verified against an equivalent PyTorch computation, and explained why
  several heads add something one large head cannot,
- derived **sinusoidal positional encodings** and demonstrated, by direct
  measurement, that self-attention is permutation-equivariant *without*
  them and is not *with* them,
- assembled **residual, pre-norm Transformer blocks** from attention and
  a position-wise feed-forward network,
- derived **causal masking** and verified its effect on which positions
  can influence which outputs, and
- **counted a Transformer block's parameters analytically** and matched
  the count exactly against a real `torch.nn` implementation.

## Introduction

Every architecture so far encodes position implicitly: a CNN's kernel
slides over a fixed spatial grid (5a), an RNN's recurrence processes
tokens strictly in order (7a). Self-attention, by contrast, computes
every output as a weighted combination of *values*, weighted only by how
well each *query* matches each *key* — nothing in that computation reads
off "this is the third token." That is a genuine gap, not a stylistic
choice, and every remaining piece of this lesson exists to work around
it: **multiple heads** let several different content-based relationships
be represented at once, **positional encodings** inject the order
information attention otherwise entirely lacks, and once ordering
matters, **causal masking** is what stops a model being trained to
predict token $t$ from silently peeking at token $t+1$.

## Setup

In [ ]:
# Fixed seeds: every stochastic step (weight init, permutation choice,
# perturbation draws) is reproducible.
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6, 4)
print("numpy:", np.__version__)
print("torch:", torch.__version__)

## Multi-Head Self-Attention

A single attention computation produces exactly one weighted
combination per query — one specific notion of "what's relevant here."
**Multi-head attention** runs $h$ independent attention computations in
parallel, each in its own, smaller $d_k = d_{\text{model}}/h$-dimensional
subspace, with its own learned projections, then concatenates and mixes
the results:

$$\text{head}_i = \text{Attention}(X W_Q^{(i)},\, X W_K^{(i)},\, X W_V^{(i)}), \qquad \text{MultiHead}(X) = \big[\text{head}_1;\dots;\text{head}_h\big] W_O.$$

Splitting one $d_{\text{model}}$-dimensional attention into $h$ heads of
dimension $d_{\text{model}}/h$ costs roughly the *same* total compute and
parameters as one full-width head (each $W_Q^{(i)}$ etc. is smaller by
exactly a factor of $h$) — multi-head attention is a **reparameterisation**,
not simply "more attention." What it buys is representational: a single
head's softmax can only express one weighting pattern per query,
however that pattern is computed, but different linguistic or structural
relationships (adjacent-word relationships, long-range dependencies,
positional patterns) generally need *different* weightings for the
*same* query — something one softmax cannot represent simultaneously,
however wide it is made, and $h$ independent softmaxes can.

In [ ]:
def softmax(x, axis=-1):
    shifted = x - x.max(axis=axis, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=axis, keepdims=True)


def multi_head_attention_np(X, Wq, Wk, Wv, Wo, num_heads):
    """X: (T, d_model). Wq, Wk, Wv, Wo: (d_model, d_model)."""
    T, d_model = X.shape
    d_k = d_model // num_heads
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    Qh = Q.reshape(T, num_heads, d_k).transpose(1, 0, 2)  # (heads, T, d_k)
    Kh = K.reshape(T, num_heads, d_k).transpose(1, 0, 2)
    Vh = V.reshape(T, num_heads, d_k).transpose(1, 0, 2)
    outputs = []
    for head in range(num_heads):
        scores = (Qh[head] @ Kh[head].T) / np.sqrt(d_k)
        weights = softmax(scores)
        outputs.append(weights @ Vh[head])
    concat = np.concatenate(outputs, axis=-1)  # (T, d_model)
    return concat @ Wo


d_model, num_heads, T = 32, 4, 6
rng = np.random.default_rng(SEED)
X_np = rng.normal(size=(T, d_model)) * 0.1
Wq_np, Wk_np, Wv_np, Wo_np = (rng.normal(size=(d_model, d_model)) * 0.1 for _ in range(4))

out_scratch = multi_head_attention_np(X_np, Wq_np, Wk_np, Wv_np, Wo_np, num_heads)

X_t, Wq_t, Wk_t, Wv_t, Wo_t = (torch.tensor(a) for a in (X_np, Wq_np, Wk_np, Wv_np, Wo_np))
Q_t, K_t, V_t = X_t @ Wq_t, X_t @ Wk_t, X_t @ Wv_t
d_k = d_model // num_heads
Qh_t = Q_t.reshape(T, num_heads, d_k).permute(1, 0, 2)
Kh_t = K_t.reshape(T, num_heads, d_k).permute(1, 0, 2)
Vh_t = V_t.reshape(T, num_heads, d_k).permute(1, 0, 2)
out_torch_heads = F.scaled_dot_product_attention(Qh_t, Kh_t, Vh_t)  # (heads, T, d_k)
out_torch = out_torch_heads.permute(1, 0, 2).reshape(T, d_model) @ Wo_t

max_diff = np.abs(out_scratch - out_torch.numpy()).max()
print(f"output shape: {out_scratch.shape}")
print(f"max abs diff vs torch.nn.functional.scaled_dot_product_attention (per head): {max_diff:.2e}")
assert max_diff < 1e-10

The from-scratch, per-head-looped implementation matches a batched
`torch.nn.functional.scaled_dot_product_attention` call (one per head,
via the exact same reshape-into-heads convention every real Transformer
implementation uses) to floating-point precision.

## Positional Encodings

Self-attention computes every output from *content* alone — a query
matches a key regardless of where either one sits in the sequence. That
makes plain self-attention **permutation-equivariant**: reorder the
input tokens, and the output for each token is exactly the same value it
would have been, just relocated to that token's new position. Reordering
`"the cat sat"` into `"cat sat the"` should mean something different to
a language model, but content-only attention has no way to represent
that difference at all — position must be injected as an explicit
signal, added directly to each token's embedding before it ever reaches
attention.

The Transformer's original choice is a fixed **sinusoidal** encoding, one
vector per position, added elementwise to the token embedding:

$$PE_{(pos,\,2i)} = \sin\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right), \qquad PE_{(pos,\,2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right).$$

Two properties make this specific choice, rather than something
simpler, worth deriving: it is bounded and smooth (unlike, say, using
the raw integer position, which would take unboundedly large values for
long sequences), and — because $\sin$ and $\cos$ of a sum expand into
linear combinations of $\sin$ and $\cos$ of the parts — $PE_{pos+k}$ is
expressible as a *fixed linear function* of $PE_{pos}$ for any offset
$k$, meaning attention weights *can* in principle learn to attend by
relative position, purely from a linear operation on the encoding. A
**learned** positional embedding (an ordinary `nn.Embedding` indexed by
position, trained like any other parameter) is simpler and often works
just as well in practice, but has no defined value for a position beyond
whatever maximum length it was trained with; the sinusoidal formula is
defined for every position, including lengths never seen during
training.

In [ ]:
def sinusoidal_positional_encoding(max_len, d_model):
    pos = np.arange(max_len)[:, None]
    i = np.arange(d_model)[None, :]
    angle_rates = 1.0 / (10000 ** (2 * (i // 2) / d_model))
    angles = pos * angle_rates
    pe = np.zeros((max_len, d_model))
    pe[:, 0::2] = np.sin(angles[:, 0::2])
    pe[:, 1::2] = np.cos(angles[:, 1::2])
    return pe


pe = sinusoidal_positional_encoding(max_len=50, d_model=32)
plt.figure(figsize=(7, 4))
plt.imshow(pe.T, cmap="RdBu", aspect="auto")
plt.xlabel("position")
plt.ylabel("encoding dimension")
plt.colorbar(label="value")
plt.title("Sinusoidal positional encoding")
plt.tight_layout()
plt.show()

Low dimensions oscillate quickly (short wavelength, changing almost
every position) and high dimensions oscillate slowly (long wavelength) —
together, every position gets a unique, bounded vector, and nearby
positions get similar vectors, exactly the smooth, order-sensitive
signal attention alone cannot generate on its own.

In [ ]:
def self_attention_np(X, Wq, Wk, Wv):
    scores = (X @ Wq) @ (X @ Wk).T / np.sqrt(X.shape[1])
    return softmax(scores) @ (X @ Wv)


T2, d = 6, 16
rng2 = np.random.default_rng(SEED)
token_embeddings = rng2.normal(size=(T2, d)) * 0.1
Wq2, Wk2, Wv2 = (rng2.normal(size=(d, d)) * 0.1 for _ in range(3))
perm = rng2.permutation(T2)

# Without positional encoding: attention on the permuted sequence should
# equal the permuted attention output on the original sequence, exactly.
out_original = self_attention_np(token_embeddings, Wq2, Wk2, Wv2)
out_permuted = self_attention_np(token_embeddings[perm], Wq2, Wk2, Wv2)
equivariance_gap_no_pe = np.abs(out_permuted - out_original[perm]).max()

# With positional encoding added before attention: a token's representation
# now depends on *which position it occupies*, not just its own identity.
pe2 = sinusoidal_positional_encoding(T2, d)
out_original_pe = self_attention_np(token_embeddings + pe2, Wq2, Wk2, Wv2)
out_permuted_pe = self_attention_np(token_embeddings[perm] + pe2, Wq2, Wk2, Wv2)
equivariance_gap_with_pe = np.abs(out_permuted_pe - out_original_pe[perm]).max()

print(f"permutation: {perm.tolist()}")
print(f"without positional encoding, max |permuted output - permuted(original output)|: {equivariance_gap_no_pe:.2e}")
print(f"with positional encoding,    max |permuted output - permuted(original output)|: {equivariance_gap_with_pe:.2e}")

Without positional encoding, permuting the input tokens produces
exactly the correspondingly permuted output — to floating-point
precision, self-attention genuinely cannot tell the difference between
"this token's content" and "this token's content, at this position."
Adding the positional encoding breaks that equivariance completely: the
same token, moved to a different position, now produces a measurably
different output, because its representation going into attention
depends on where it sits, not only on what it is.

## The Transformer Block

One Transformer block wraps multi-head attention and a position-wise
feed-forward network in **residual connections** and **layer
normalisation**, in the widely used *pre-norm* arrangement:

$$x' = x + \text{MultiHead}(\text{LayerNorm}(x)), \qquad x'' = x' + \text{FFN}(\text{LayerNorm}(x')), \qquad \text{FFN}(z) = W_2\, \sigma(W_1 z + b_1) + b_2.$$

The residual ("+x") term is the same idea as 7a's LSTM cell-state
path, applied to depth instead of time: it gives gradients (and, in the
forward direction, the original representation itself) a path around
each sub-layer that requires no learned transformation to preserve, so
stacking many blocks does not by itself have to fight the vanishing
gradients an equally deep network of ordinary layers would. Layer
normalisation rescales each token's activations to zero mean and unit
variance across the feature dimension (independent of batch size, unlike
batch normalisation), which keeps the scale of what every sub-layer
receives roughly consistent regardless of how large its inputs' raw
activations happen to be — stabilising training in exactly the deep,
many-block stacks this architecture is built to support.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))

    def forward(self, x, attn_mask=None):
        normed = self.ln1(x)
        attn_out, _ = self.attn(normed, normed, normed, attn_mask=attn_mask, need_weights=False)
        x = x + attn_out
        x = x + self.ffn(self.ln2(x))
        return x


D_MODEL, NUM_HEADS, D_FF = 32, 4, 64
block = TransformerBlock(D_MODEL, NUM_HEADS, D_FF)
sample_input = torch.randn(1, 10, D_MODEL)
sample_output = block(sample_input)
print(f"input shape: {tuple(sample_input.shape)}, output shape: {tuple(sample_output.shape)}")

## Causal Masking

Training a language model with teacher forcing computes every output
position in parallel from the same input sequence — which means, unless
prevented, self-attention would let the prediction for position $t$
attend directly to position $t+1$ and beyond: seeing the very answer it
is being trained to predict. **Causal masking** prevents this by forcing
every attention score for a "future" key to $-\infty$ before the
softmax, so its weight after softmax is exactly zero:

$$\text{mask}_{i,j} = \begin{cases} 0 & j \le i \\ -\infty & j > i \end{cases}, \qquad \text{scores}' = \text{scores} + \text{mask}.$$

The result is a strictly lower-triangular pattern of allowed
attention — the same shape appears in every autoregressive Transformer,
independent of what the model is trained on.

In [ ]:
def causal_mask(T):
    mask = np.triu(np.ones((T, T)), k=1).astype(bool)  # True where j > i (future positions)
    return mask


T3 = 8
mask = causal_mask(T3)
plt.figure(figsize=(4.5, 4))
plt.imshow(~mask, cmap="Greys", vmin=0, vmax=1)
plt.xlabel("key position (attended to)")
plt.ylabel("query position (attending)")
plt.title("Causal mask (white = allowed)")
plt.tight_layout()
plt.show()

In [ ]:
def masked_self_attention_np(X, Wq, Wk, Wv, mask=None):
    scores = (X @ Wq) @ (X @ Wk).T / np.sqrt(X.shape[1])
    if mask is not None:
        scores = np.where(mask, -np.inf, scores)
    return softmax(scores) @ (X @ Wv)


rng3 = np.random.default_rng(SEED)
X3 = rng3.normal(size=(T3, d)) * 0.1
Wq3, Wk3, Wv3 = (rng3.normal(size=(d, d)) * 0.1 for _ in range(3))
X3_perturbed = X3.copy()
X3_perturbed[-1] += rng3.normal(size=d)  # change only the LAST (future-most) position

out_masked_orig = masked_self_attention_np(X3, Wq3, Wk3, Wv3, mask)
out_masked_pert = masked_self_attention_np(X3_perturbed, Wq3, Wk3, Wv3, mask)
out_unmasked_orig = masked_self_attention_np(X3, Wq3, Wk3, Wv3, mask=None)
out_unmasked_pert = masked_self_attention_np(X3_perturbed, Wq3, Wk3, Wv3, mask=None)

# Every position except the last one was untouched by the perturbation.
masked_diff = np.abs(out_masked_orig[:-1] - out_masked_pert[:-1]).max()
unmasked_diff = np.abs(out_unmasked_orig[:-1] - out_unmasked_pert[:-1]).max()
print(f"causally masked: max change at earlier positions from a later-position perturbation: {masked_diff:.2e}")
print(f"unmasked:        max change at earlier positions from a later-position perturbation: {unmasked_diff:.2e}")

With the causal mask, perturbing only the final token leaves every
earlier position's output completely unchanged — to floating-point
precision, an earlier query's attention weights never included the
perturbed key at all. Without the mask, the same perturbation changes
every earlier position's output too, because unmasked attention lets
every query see every key regardless of order — exactly the
future-peeking the mask exists to prevent.

## Parameter Counting

Every learned matrix in one Transformer block can be counted by hand.
Multi-head attention's four projections ($W_Q, W_K, W_V, W_O$) are each
$d_{\text{model}} \times d_{\text{model}}$ with a bias of length
$d_{\text{model}}$ (splitting into heads reshapes these same matrices —
it does not add parameters):

$$N_{\text{attn}} = 4\,(d_{\text{model}}^2 + d_{\text{model}}).$$

The feed-forward network has one $d_{\text{model}} \times d_{\text{ff}}$
and one $d_{\text{ff}} \times d_{\text{model}}$ matrix, each with its own
bias:

$$N_{\text{ffn}} = 2\, d_{\text{model}}\, d_{\text{ff}} + d_{\text{ff}} + d_{\text{model}}.$$

Each of the two layer norms contributes a learned scale and shift per
feature:

$$N_{\text{ln}} = 2 \times 2\, d_{\text{model}} = 4\, d_{\text{model}}.$$

$$N_{\text{block}} = N_{\text{attn}} + N_{\text{ffn}} + N_{\text{ln}}.$$

In [ ]:
def transformer_block_param_count(d_model, d_ff):
    n_attn = 4 * (d_model ** 2 + d_model)
    n_ffn = 2 * d_model * d_ff + d_ff + d_model
    n_ln = 4 * d_model
    return n_attn + n_ffn + n_ln


formula_count = transformer_block_param_count(D_MODEL, D_FF)
actual_count = sum(p.numel() for p in block.parameters())
print(f"formula predicts: {formula_count} parameters")
print(f"actual nn.Module: {actual_count} parameters")
assert formula_count == actual_count

The hand-derived formula matches the real implementation's parameter
count exactly — every weight and bias in a Transformer block is
accounted for, with nothing hidden in a library default.

## Key Takeaways

- **Multi-head attention reparameterises one wide attention into several
  narrow ones** at roughly the same total cost, letting the model
  represent several different content-based relationships for the same
  query simultaneously — verified against a batched
  `scaled_dot_product_attention` call to floating-point precision.
- **Self-attention is permutation-equivariant without positional
  information, measured exactly**: permuting inputs permutes outputs
  identically with no positional encoding added, and that exact
  equivariance breaks completely once a sinusoidal positional encoding
  is added — position has to be injected, it is not implicit in the
  operator.
- **A Transformer block is attention and a feed-forward network wrapped
  in residual connections and layer norm** — the same additive,
  gradient-preserving idea as 7a's LSTM cell state, applied to depth
  rather than time.
- **Causal masking makes an exact, verifiable guarantee**: perturbing a
  future token changes no earlier position's output at all, confirmed
  directly, while the identical perturbation changes every earlier
  output when the mask is removed.
- **A Transformer block's parameter count is fully predictable by hand**
  — the derived formula matched a real `torch.nn` implementation's
  parameter count exactly.